# Research: Stock Recommendations & Data Mining (Unified Version)

This notebook contains the unified data mining, technical analysis, and vectorization logic.
It allows you to switch between local CSV caching, local database access, or live data mining.


In [ ]:
!pip install yfinance matplotlib seaborn scipy lxml html5lib requests tqdm

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import math
import time
import os
import requests
from scipy.stats import norm
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set up matplotlib style for financial charts
plt.style.use('default')
%matplotlib inline

####################################################
# CONFIGURATION FLAG: Select your data source here #
####################################################
# Options: 'CSV', 'DB', 'MINE_AND_CACHE'
DATA_SOURCE_MODE = 'CSV'

# Local files / database paths
PRICE_FILE = 'sp1500_price_matrix.csv'
VOLUME_FILE = 'sp1500_volume_matrix.csv'
MASTER_FILE = 'sp1500_master_research_dataset.csv'
DB_PATH = 'backend/stock_recommender.db'

if DATA_SOURCE_MODE == 'DB':
    conn = sqlite3.connect(DB_PATH)


## 1. Massive Data Extraction (S&P 1500 Universe)
We scrape the massive universe of 1500+ stocks safely, then pull their max historical daily closes. (Includes CSV Caching to save time on reruns!)

In [ ]:
if DATA_SOURCE_MODE == 'CSV':
    if os.path.exists(PRICE_FILE) and os.path.exists(VOLUME_FILE):
        print("\n=> [CSV Mode] Loading directly from CSVs to save time...")
        price_matrix = pd.read_csv(PRICE_FILE, index_col=0, parse_dates=True)
        volume_matrix = pd.read_csv(VOLUME_FILE, index_col=0, parse_dates=True)
    else:
        print("\n=> [CSV Mode] Error: CSV files not found. Please run with 'MINE_AND_CACHE' first.")

elif DATA_SOURCE_MODE == 'DB':
    print("\n=> [DB Mode] Loading from local SQLite Database...")
    # Note: Assuming 'daily_quotes' or 'asset_prices' table structure (date, ticker, close, volume)
    # Adjust query to sync with your actual backend schema!
    try:
        # Execute query for daily prices
        df = pd.read_sql("SELECT date, ticker, close, volume FROM asset_prices", conn, parse_dates=['date'])
        
        # Pivot to create matrices
        price_matrix = df.pivot(index='date', columns='ticker', values='close')
        volume_matrix = df.pivot(index='date', columns='ticker', values='volume')
        
        price_matrix = price_matrix.resample('D').ffill()
        volume_matrix = volume_matrix.resample('D').ffill()
        print(f"\n=> Loaded DB data for {len(price_matrix.columns)} tickers.")
    except Exception as e:
        print("\n=> [DB Mode] Error reading from DB:", e)
        print("Please ensure the query matches your schema.")

elif DATA_SOURCE_MODE == 'MINE_AND_CACHE':
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) stock-research-toolkit/1.0'}

    def scrape_wiki_tickers(url, table_idx, ticker_col):
        try:
            res = requests.get(url, headers=headers)
            table = pd.read_html(res.text)[table_idx]
            tickers = table[ticker_col].str.replace('.', '-', regex=False).tolist()
            time.sleep(1)
            return tickers
        except:
            print(f"Failed to scrape {url}")
            return []

    print("Scraping S&P 500 (Large-Cap)...")
    sp500 = scrape_wiki_tickers('https://en.wikipedia.org/wiki/List_of_S%26P_500_companies', 0, 'Symbol')

    print("Scraping S&P 400 (Mid-Cap)...")
    sp400 = scrape_wiki_tickers('https://en.wikipedia.org/wiki/List_of_S%26P_400_companies', 0, 'Symbol')

    print("Scraping S&P 600 (Small-Cap)...")
    sp600 = scrape_wiki_tickers('https://en.wikipedia.org/wiki/List_of_S%26P_600_companies', 0, 'Symbol')

    print("Scraping Nasdaq 100...")
    ndx = scrape_wiki_tickers('https://en.wikipedia.org/wiki/Nasdaq-100', 4, 'Ticker')

    EQUITY_ETFS = ["VOO", "QQQ", "VTI", "VXUS", "VGT", "ARKK", "VNQ", "VWO"]
    BOND_ETFS = ["BND", "SGOV", "TLT", "TIPS"]

    # Combine all identified tickers into a 1500+ Mega-list
    ALL_TICKERS = list(set(sp500 + sp400 + sp600 + ndx + EQUITY_ETFS + BOND_ETFS))
    print(f"\nTotal Unique S&P 1500+ Universe Assembled: {len(ALL_TICKERS)}")

    print("\n=> Fetching 'max' history from Yahoo Finance (Warning: Downloading millions of data points will take a few minutes)...")
    raw_data = yf.download(ALL_TICKERS, period="max", group_by="ticker", auto_adjust=True, progress=True)

    # Build matrices natively using cross-section to avoid DataFrame fragmentation
    if isinstance(raw_data.columns, pd.MultiIndex):
        price_matrix = raw_data.xs('Close', axis=1, level=1).copy()
        volume_matrix = raw_data.xs('Volume', axis=1, level=1).copy()
    else:
        price_matrix = pd.DataFrame({ALL_TICKERS[0]: raw_data['Close']})
        volume_matrix = pd.DataFrame({ALL_TICKERS[0]: raw_data['Volume']})
        
    # Forward fill weekends/holidays up to today
    price_matrix = price_matrix.resample('D').ffill()
    volume_matrix = volume_matrix.resample('D').ffill()
    
    # Save for future runs
    price_matrix.to_csv(PRICE_FILE)
    volume_matrix.to_csv(VOLUME_FILE)
    print("\n=> Cached matrix data to CSVs for future runs.")

daily_returns = price_matrix.pct_change().dropna(how='all')
correlation_matrix = daily_returns.corr()

print(f"\nSuccessfully loaded historical price matrix: {price_matrix.shape[0]} Days x {price_matrix.shape[1]} Tickers")


## 2. Vectorized Historical Technical Indicators
Compute sliding-window indicators (MACD, RSI, Bollinger Bands, Volatility) for the entire dataset matrix instantly using pure Pandas vectorization.

In [ ]:
print("Calculating Historical Technical Indicators (MACD, RSI, BB, Volatility) for 1500+ Stocks...")

# EMA and SMA for MACD logic
ema_12 = price_matrix.ewm(span=12, adjust=False).mean()
ema_26 = price_matrix.ewm(span=26, adjust=False).mean()
macd_line = ema_12 - ema_26
macd_signal = macd_line.ewm(span=9, adjust=False).mean()

# RSI (14-day) Sliding window across entire matrix natively
delta = price_matrix.diff()
gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
rs = gain / loss
rsi_14 = 100 - (100 / (1 + rs))

# Bollinger Bands (20-day, 2 std)
bb_std = price_matrix.rolling(window=20).std()
bb_upper = price_matrix.rolling(window=20).mean() + (bb_std * 2)
bb_lower = price_matrix.rolling(window=20).mean() - (bb_std * 2)

# Volatilities (Annualised) moving windows
vol_30d = daily_returns.rolling(window=30).std() * np.sqrt(252)
vol_90d = daily_returns.rolling(window=90).std() * np.sqrt(252)
vol_1yr = daily_returns.rolling(window=252).std() * np.sqrt(252)
vol_3yr = daily_returns.rolling(window=252*3).std() * np.sqrt(252)
vol_5yr = daily_returns.rolling(window=252*5).std() * np.sqrt(252)

# Rolling Returns
return_1yr = price_matrix.pct_change(periods=252)
return_3yr = price_matrix.pct_change(periods=252*3)
return_5yr = price_matrix.pct_change(periods=252*5)

# Maximum Drawdown (1-year rolling)
rolling_max_1yr = price_matrix.rolling(window=252, min_periods=1).max()
drawdown_1yr = (price_matrix - rolling_max_1yr) / rolling_max_1yr
max_drawdown_1yr = drawdown_1yr.rolling(window=252, min_periods=1).min()

print("Historical feature matrices generated instantaneously!")

## 3. Build Master Research DataFrame (Heavy API Loop)
We will now loop through all 1500+ tickers to query fundamental info & Options Greeks. 
**NOTE:** This automatically saves a giant CSV caching the results, so you never have to waste an hour running this twice!

In [ ]:
if DATA_SOURCE_MODE == 'CSV':
    if os.path.exists(MASTER_FILE):
        print("\n=> [CSV Mode] Found cached Master Research Dataset! Loading directly from CSV...")
        master_research_df = pd.read_csv(MASTER_FILE, index_col=0)
    else:
        print("\n=> [CSV Mode] Error: Master Dataset CSV not found.")

elif DATA_SOURCE_MODE == 'DB':
    print("\n=> [DB Mode] Accessing specific features via Database...")
    try:
        master_research_df = pd.read_sql("SELECT * FROM assets", conn, index_col='ticker')
        print(f"\n=> Loaded backend Features DB data for {len(master_research_df)} assets.")
    except Exception as e:
        print("\n=> [DB Mode] Error reading from DB:", e)

elif DATA_SOURCE_MODE == 'MINE_AND_CACHE':
    print("\n=> [MINE_AND_CACHE Mode] Beginning the heavy 1-hour API extraction loop...")
    RISK_FREE_RATE = 0.045
    company_data = []

    # If you don't have an hour to wait natively, change ALL_TICKERS to ALL_TICKERS[:50] to test a smaller subset!
    for ticker in tqdm(ALL_TICKERS, desc="Fetching Fundamental & Options Data"):
        try:
            t_obj = yf.Ticker(ticker)
            info = t_obj.info
            
            # We only process valid stocks that return info
            if 'symbol' not in info and 'shortName' not in info:
                continue
                
            company_record = {
                'ticker': ticker,
                'sector': info.get('sector', 'Unknown'),
                'industry': info.get('industry', 'Unknown'),
                'market_cap': info.get('marketCap', 0),
                'pe_ratio': info.get('trailingPE', np.nan),
                'beta': info.get('beta', np.nan),
                'dividend_yield': info.get('dividendYield', 0), # NOTE: yfinance auto_adjust=True reinvests dividends into historical prices,
                
                # Detailed Company Information
                'long_description': info.get('longBusinessSummary', ''),
                'website': info.get('website', ''),
                'city': info.get('city', ''),
                'state': info.get('state', ''),
                'country': info.get('country', ''),
                'full_time_employees': info.get('fullTimeEmployees', np.nan),
                
                # Financial / Quarterly Reports Context (TTM Metrics)
                'total_revenue': info.get('totalRevenue', np.nan),
                'revenue_growth': info.get('revenueGrowth', np.nan),
                'gross_margins': info.get('grossMargins', np.nan),
                'operating_margins': info.get('operatingMargins', np.nan),
                'ebitda': info.get('ebitda', np.nan),
                'free_cashflow': info.get('freeCashflow', np.nan),
                'total_cash': info.get('totalCash', np.nan),
                'total_debt': info.get('totalDebt', np.nan)
            }
            
            # Options Greeks (Black-Scholes Approximation)
            try:
                exps = t_obj.options
                if exps:
                    nearest_exp = exps[0]
                    opt_chain = t_obj.option_chain(nearest_exp)
                    calls = opt_chain.calls
                    
                    if not calls.empty and 'currentPrice' in info:
                        current_price = info['currentPrice']
                        atm_call = calls.iloc[(calls['strike'] - current_price).abs().argsort()[:1]].iloc[0]
                        
                        S = current_price
                        K = atm_call['strike']
                        T = 30 / 365.0  # Approx 1 month
                        r = RISK_FREE_RATE
                        sigma = atm_call.get('impliedVolatility', 0.2)
                        
                        if sigma > 0:
                            d1 = (math.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * math.sqrt(T))
                            delta = norm.cdf(d1)
                            gamma = norm.pdf(d1) / (S * sigma * math.sqrt(T))
                            vega = S * norm.pdf(d1) * math.sqrt(T) / 100
                            
                            company_record.update({
                                'atm_strike': K,
                                'implied_vol': sigma,
                                'delta': delta,
                                'gamma': gamma,
                                'vega': vega
                            })
            except:
                pass # Options chain failed or not available
            
            company_data.append(company_record)
            time.sleep(0.3)  # Anti-Rate Limit Delay (crucial for 1500+ loops)
        except Exception as e:
            pass # Ignore individual failed ticker fetches
    
    company_df = pd.DataFrame(company_data).set_index('ticker')
    
    current_metrics = pd.DataFrame({
        'latest_price': price_matrix.iloc[-1],
        'avg_30d_volume': volume_matrix.rolling(30).mean().iloc[-1],
        'rsi_14': rsi_14.iloc[-1],
        'macd': macd_line.iloc[-1],
        'macd_signal': macd_signal.iloc[-1],
        'volatility_30d': vol_30d.iloc[-1],
        'volatility_1yr': vol_1yr.iloc[-1],
        'return_1yr': return_1yr.iloc[-1],
        'return_3yr': return_3yr.iloc[-1] if 'return_3yr' in locals() else np.nan,
        'return_5yr': return_5yr.iloc[-1] if 'return_5yr' in locals() else np.nan,
        'volatility_3yr': vol_3yr.iloc[-1] if 'vol_3yr' in locals() else np.nan,
        'volatility_5yr': vol_5yr.iloc[-1] if 'vol_5yr' in locals() else np.nan,
        'max_drawdown_1yr': max_drawdown_1yr.iloc[-1] if 'max_drawdown_1yr' in locals() else np.nan
    })
    
    master_research_df = current_metrics.join(company_df)
    
    # Save the fully joined dataframe securely to a CSV to avoid ever having to run this loop again tomorrow
    master_research_df.to_csv(MASTER_FILE)

if 'master_research_df' in locals():
    print(f"\nMaster Research Dataset Successfully Loaded: {master_research_df.shape[0]} Stocks x {master_research_df.shape[1]} Features")
    display(master_research_df.head())


## 4. Giant Universe Recommender Engine Research
We can now find alternative stocks accurately utilizing 1500+ equities and precise metrics (RSI, Options Implied Vol, etc).

In [ ]:
def find_similar_stocks(target_ticker, top_n=5, method="correlation"):
    if target_ticker not in correlation_matrix.columns:
        return f"Ticker {target_ticker} not in database."
        
    if method == "correlation":
        corrs = correlation_matrix[target_ticker].sort_values(ascending=False)
        return pd.DataFrame({'Correlation': corrs[corrs.index != target_ticker].head(top_n)})
        
    elif method == "features":
        # Use the mega-dataset features here!
        feats = master_research_df[['rsi_14', 'macd', 'volatility_30d', 'return_1yr']].dropna()
        if target_ticker not in feats.index:
            return f"Missing features for {target_ticker}"
        
        normalized = (feats - feats.mean()) / feats.std()
        target_vec = normalized.loc[target_ticker]
        
        distances = np.sqrt(((normalized - target_vec)**2).sum(axis=1))
        distances = distances.sort_values()
        similar = distances[distances.index != target_ticker].head(top_n)
        
        return master_research_df.loc[similar.index, ['sector', 'latest_price', 'rsi_14', 'implied_vol']]

print("Top 5 correlated S&P 1500+ stocks to AAPL:")
display(find_similar_stocks("AAPL", method="correlation"))

print("\nTop 5 stocks with identical MACD/RSI/Volatility technical profiles to MSFT:")
display(find_similar_stocks("MSFT", method="features"))

## 4b. Dynamic Multi-Horizon Asset Categorization & Scoring
This section extracts core performance statistics (Return and Variance) over the **complete available history** of every asset. 
It applies an **Exponential Moving Decay** strictly to Annual calculations, automatically placing a significantly higher ranking weight on recent years' performance (e.g., 2024 matters far more than 1999) without discarding long-term data.

We then execute tailored selection queries to extract the perfect tickers for the Short, Medium, and Long Term buckets of our Custom Portfolio.

In [ ]:
import pandas as pd
import numpy as np

print("Aggregating historical matrices into Annual blocks...")
# 1. Resample matrices to precise Annual Blocks
annual_prices = price_matrix.resample('YE').last()
annual_returns = annual_prices.pct_change().dropna(how='all')

# 2. To get accurate annual variance, we group the daily returns by year and calculate variance multiplied by 252
annual_variance = daily_returns.groupby(daily_returns.index.year).var() * 252

# Align indices just in case
annual_returns.index = annual_returns.index.year
annual_returns = annual_returns[annual_returns.index.isin(annual_variance.index)]

# 3. Apply Exponential Weighting (Decay) across all available years. 
# A span of 10 gives significant weight to the last ~5 years, but still factors in long-term history smoothly.
# .iloc[-1] grabs the final compounded weighted value for the present day.
weighted_return = annual_returns.ewm(span=10).mean().iloc[-1]
weighted_variance = annual_variance.ewm(span=10).mean().iloc[-1]

# 4. Compile Scoring DataFrame
scoring_df = pd.DataFrame({
    'weighted_annual_return': weighted_return,
    'weighted_annual_variance': weighted_variance
}).dropna() # Drop entirely dead tickers

# Calculate baseline Sharpe Ratio (Risk-Adjusted Return)
# Assuming a ~4% risk-free rate structurally
RISK_FREE_RATE = 0.04
scoring_df['raw_sharpe'] = (scoring_df['weighted_annual_return'] - RISK_FREE_RATE) / np.sqrt(scoring_df['weighted_annual_variance'])

print(f"Computed weighted statistics for {len(scoring_df)} individual assets.\n")

# ---------------------------------------------------------
# Dynamic Selection Queries (Using Time-Horizon Penalty Formula)
# ---------------------------------------------------------

# Formula: Score = Expected Return - [ (30 - Horizon_Years) / 30 ] * Variance
# This elegantly ensures that a 30-year horizon has 0 variance penalty (pure growth),
# while a 1-year horizon has a ~96% variance penalty (extreme safety).

def get_top_assets_for_horizon(horizon_years, top_n=5):
    penalty_weight = (30 - horizon_years) / 30.0
    col_name = f'score_{int(horizon_years)}yr'
    scoring_df[col_name] = scoring_df['weighted_annual_return'] - (penalty_weight * scoring_df['weighted_annual_variance'])
    
    return scoring_df.sort_values(by=col_name, ascending=False).head(top_n)

# 1. SHORT TERM PORTFOLIO (1-Year Goal)
short_term_picks = get_top_assets_for_horizon(1)
print("=== TOP 5 SHORT-TERM ASSETS (1-Year Goal) ===")
display(master_research_df.loc[short_term_picks.index, ['sector', 'latest_price']].join(short_term_picks[['weighted_annual_return', 'weighted_annual_variance', 'score_1yr']]))

# 2. MEDIUM TERM PORTFOLIO (15-Year Goal)
medium_term_picks = get_top_assets_for_horizon(15)
print("\n=== TOP 5 MEDIUM-TERM ASSETS (15-Year Goal) ===")
display(master_research_df.loc[medium_term_picks.index, ['sector', 'latest_price']].join(medium_term_picks[['weighted_annual_return', 'weighted_annual_variance', 'score_15yr']]))

# 3. LONG TERM PORTFOLIO (30-Year Goal)
long_term_picks = get_top_assets_for_horizon(30)
print("\n=== TOP 5 LONG-TERM ASSETS (30-Year Goal) ===")
display(master_research_df.loc[long_term_picks.index, ['sector', 'latest_price']].join(long_term_picks[['weighted_annual_return', 'weighted_annual_variance', 'score_30yr']]))

## 5. Goal-Based Monte Carlo Simulation (Historically Calibrated)
This simulation is calibrated using **real historical mean returns and volatility** extracted
directly from the price matrix — NOT hardcoded assumptions.

Capital allocation follows the same 4-bucket rule as Section 6:
- **0–1 yr goals** → Cash (flat ~3% annual)
- **1–5 yr goals** → Bonds (VBTIX-calibrated)
- **5–15 yr goals** → 70% S&P 500 / 30% Bonds
- **15–30 yr goals + surplus** → 100% S&P 500

Each portfolio shows exactly **3 lines**: mean trajectory, 5th-percentile band, 95th-percentile band.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# ── Calibrate from real historical data ──────────────────────────
SP500_TICKER  = '^GSPC' if '^GSPC' in price_matrix.columns else price_matrix.columns[0]
BOND_TICKER   = 'VBTIX'  if 'VBTIX'  in price_matrix.columns else None

sp500_daily   = price_matrix[SP500_TICKER].pct_change().dropna()
SP500_MU      = sp500_daily.mean() * 252         # annualised mean
SP500_SIGMA   = sp500_daily.std()  * np.sqrt(252) # annualised vol

if BOND_TICKER:
    bond_daily  = price_matrix[BOND_TICKER].pct_change().dropna()
    BOND_MU     = bond_daily.mean() * 252
    BOND_SIGMA  = bond_daily.std()  * np.sqrt(252)
else:
    BOND_MU, BOND_SIGMA = 0.04, 0.06  # fallback

CASH_MU, CASH_SIGMA = 0.03, 0.005   # near risk-free

print(f"Calibrated params:  SP500 μ={SP500_MU:.2%}  σ={SP500_SIGMA:.2%}")
print(f"                    Bond  μ={BOND_MU:.2%}  σ={BOND_SIGMA:.2%}")

# ── Portfolio configuration ───────────────────────────────────────
GOALS_CONFIG = {1: 20000, 5: 200000, 15: 50000}   # {year: amount}
STARTING_CAPITAL    = 250_000
MONTHLY_CONTRIBUTION = 1_000
HORIZON_YEARS        = 30
N_SIMS               = 1000

ASSET_PARAMS = {
    'Cash':  {'mu': CASH_MU,  'sigma': CASH_SIGMA},
    'Bonds': {'mu': BOND_MU,  'sigma': BOND_SIGMA},
    'SP500': {'mu': SP500_MU, 'sigma': SP500_SIGMA},
}

def allocate_personalized(start_cap, goals):
    """Allocate starting capital across buckets based on goal horizon."""
    alloc = {'Cash': 0.0, 'Bonds': 0.0, 'SP500': 0.0}
    remaining = float(start_cap)
    for yr, amt in sorted(goals.items()):
        if remaining <= 0:
            break
        funding = min(float(amt), remaining)
        if yr <= 1:
            alloc['Cash']  += funding
        elif yr <= 5:
            alloc['Bonds'] += funding
        elif yr <= 15:
            alloc['SP500'] += funding * 0.70
            alloc['Bonds'] += funding * 0.30
        else:
            alloc['SP500'] += funding
        remaining -= funding
    alloc['SP500'] += remaining   # surplus → long-term growth
    return alloc

def run_mc(is_personalized, n_sims=N_SIMS, years=HORIZON_YEARS, goals=GOALS_CONFIG):
    months = years * 12
    trajectories = []

    for _ in range(n_sims):
        if is_personalized:
            alloc = allocate_personalized(STARTING_CAPITAL, goals)
        else:
            alloc = {'Cash': 0.0, 'Bonds': 0.0, 'SP500': float(STARTING_CAPITAL)}

        traj = [sum(alloc.values())]
        for m in range(1, months + 1):
            year_idx = (m - 1) // 12 + 1
            for asset, val in alloc.items():
                if val > 0:
                    p  = ASSET_PARAMS[asset]
                    mu_m    = p['mu'] / 12
                    sigma_m = p['sigma'] / np.sqrt(12)
                    alloc[asset] = val * (1 + np.random.normal(mu_m, sigma_m))

            # Monthly contribution → longest-horizon bucket
            if year_idx != 15:
                alloc['SP500'] += MONTHLY_CONTRIBUTION

            # Goal liquidation at year-end months
            if m % 12 == 0 and year_idx in goals:
                needed = float(goals[year_idx])
                for asset in ['Cash', 'Bonds', 'SP500']:
                    if needed <= 0:
                        break
                    drawn = min(alloc[asset], needed)
                    alloc[asset] -= drawn
                    needed -= drawn
                if needed > 0:
                    alloc = {'Cash': 0.0, 'Bonds': 0.0, 'SP500': 0.0}  # structural wipe

            traj.append(sum(alloc.values()))
        trajectories.append(traj)

    arr = np.array(trajectories)   # shape (n_sims, months+1)
    return arr

print(f"\nRunning {N_SIMS} Monte Carlo simulations each for Baseline and Personalized portfolio...")
base_arr = run_mc(is_personalized=False)
pers_arr = run_mc(is_personalized=True)

months_axis = np.arange(HORIZON_YEARS * 12 + 1)

def plot_mc_bands(arr, color, label):
    mean   = arr.mean(axis=0)
    p5     = np.percentile(arr, 5,  axis=0)
    p95    = np.percentile(arr, 95, axis=0)
    plt.plot(months_axis, mean,  color=color,  linewidth=2,   label=f'{label} — Mean')
    plt.plot(months_axis, p5,    color=color,  linewidth=1,   linestyle='--', alpha=0.6, label=f'{label} — 5th pct')
    plt.plot(months_axis, p95,   color=color,  linewidth=1,   linestyle='--', alpha=0.6, label=f'{label} — 95th pct')
    plt.fill_between(months_axis, p5, p95, color=color, alpha=0.08)

fig, ax = plt.subplots(figsize=(14, 6))
plot_mc_bands(base_arr, '#e74c3c', 'Baseline (100% S&P 500)')
plot_mc_bands(pers_arr, '#2980b9', 'Personalized (Goal-Bucketed)')

# Mark goal events
for yr, amt in GOALS_CONFIG.items():
    plt.axvline(yr * 12, color='gray', linestyle=':', alpha=0.7)
    plt.text(yr * 12 + 1, plt.ylim()[1] * 0.9, f'Y{yr}\n${amt/1000:.0f}k', fontsize=8, color='gray')

plt.axhline(1_000_000, color='green', linestyle='--', alpha=0.5, label='$1M Target')
plt.title('Monte Carlo Portfolio Simulation — Historically Calibrated')
plt.xlabel('Months')
plt.ylabel('Portfolio Value ($)')
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

print(f"\n--- Mean 30-Year Terminal Values ---")
print(f"Baseline:     ${base_arr[:,-1].mean():>12,.2f}  (std: ${base_arr[:,-1].std():,.0f})")
print(f"Personalized: ${pers_arr[:,-1].mean():>12,.2f}  (std: ${pers_arr[:,-1].std():,.0f})")
print(f"Baseline hit $1M:     {(base_arr[:,-1] >= 1e6).mean()*100:.1f}%")
print(f"Personalized hit $1M: {(pers_arr[:,-1] >= 1e6).mean()*100:.1f}%")


## 5b. Monte Carlo Prediction Accuracy Study (MSE vs Historical)
This section asks: **how accurately does Monte Carlo predict actual future prices?**

For each asset we:
1. Train on pre-2019 daily returns to calibrate MC parameters.
2. Run 200 MC forward simulations over 2019–2024.
3. Compare MC median path against the **actual price path** using **MSE on daily returns**.
4. Compute average MSE separately for **ETFs vs individual stocks**.

The final auto-conclusion cell makes a data-driven recommendation on Monte Carlo's reliability.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

HOLDOUT_START = '2019-01-01'
N_MC_EVAL     = 200   # MC paths per asset for evaluation

# Known ETF tickers in our universe
KNOWN_ETFS = {'VOO','QQQ','VTI','VXUS','VGT','ARKK','VNQ','VWO',
              'BND','SGOV','TLT','TIPS','VBTIX','SPY','IWM','GLD',
              'SLV','XLK','XLF','XLE','XLV','XLU','XLP','XLI','XLB'}

# Split price matrix
pm_train = price_matrix[price_matrix.index < HOLDOUT_START]
pm_test  = price_matrix[price_matrix.index >= HOLDOUT_START]

# Use a representative sample: 50 ETFs + 50 stocks (or all if fewer)
all_tickers = price_matrix.columns.tolist()
etf_tickers  = [t for t in all_tickers if t in KNOWN_ETFS and t in pm_train.columns and t in pm_test.columns]
stock_tickers = [t for t in all_tickers if t not in KNOWN_ETFS and t in pm_train.columns and t in pm_test.columns]

sample_etfs   = etf_tickers[:50]
sample_stocks = stock_tickers[:50]
sample_all    = sample_etfs + sample_stocks

print(f"Evaluating MC accuracy on {len(sample_etfs)} ETFs and {len(sample_stocks)} stocks...")
print(f"Training period: pre-{HOLDOUT_START}  |  Test: {HOLDOUT_START} – present\n")

mse_results = []

for ticker in tqdm(sample_all, desc='MC accuracy evaluation'):
    try:
        train_returns = pm_train[ticker].pct_change().dropna()
        test_returns  = pm_test[ticker].pct_change().dropna()

        if len(train_returns) < 252 or len(test_returns) < 50:
            continue

        mu    = train_returns.mean()
        sigma = train_returns.std()
        T     = len(test_returns)

        # Run N_MC_EVAL Monte Carlo paths of length T (daily steps)
        mc_paths = np.zeros((N_MC_EVAL, T))
        for i in range(N_MC_EVAL):
            shocks = np.random.normal(mu, sigma, T)
            # Cumulative return from a base of 1.0
            mc_paths[i] = np.cumprod(1 + shocks) - 1

        mc_median_path = np.median(mc_paths, axis=0)

        # Actual cumulative return
        actual_path = (1 + test_returns.values).cumprod() - 1  # normalize to same scale

        # MSE on cumulative return paths (normalised, so MSE is unit-free)
        mse = np.mean((mc_median_path - actual_path) ** 2)

        mse_results.append({
            'ticker': ticker,
            'is_etf': ticker in KNOWN_ETFS,
            'mse': mse,
            'train_years': len(train_returns) / 252,
            'test_days': T
        })
    except Exception:
        continue

mse_df = pd.DataFrame(mse_results)

etf_mse   = mse_df[mse_df['is_etf']  == True]['mse']
stock_mse = mse_df[mse_df['is_etf']  == False]['mse']

print(f"=== Monte Carlo Prediction Accuracy ===")
print(f"ETF   avg MSE: {etf_mse.mean():.6f}  (median: {etf_mse.median():.6f}, n={len(etf_mse)})")
print(f"Stock avg MSE: {stock_mse.mean():.6f}  (median: {stock_mse.median():.6f}, n={len(stock_mse)})")

# Box plot comparison
fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot([etf_mse.clip(upper=etf_mse.quantile(0.95)),
            stock_mse.clip(upper=stock_mse.quantile(0.95))],
           labels=['ETFs', 'Stocks'],
           patch_artist=True,
           boxprops=dict(facecolor='#d5e8f7'))
ax.set_title('Monte Carlo Prediction MSE:\nETFs vs Individual Stocks (2019–2024 holdout)')
ax.set_ylabel('MSE (cumulative return path, lower = better)')
plt.tight_layout()
plt.show()

# ── Auto-conclusion ──────────────────────────────────────────────
ratio = etf_mse.mean() / stock_mse.mean() if stock_mse.mean() > 0 else 1.0

print("\n=== AUTO-CONCLUSION: Should We Use Monte Carlo or Historical Backtesting? ===\n")
if ratio < 0.6:
    print("Monte Carlo is substantially MORE accurate for ETFs than individual stocks.")
    print("RECOMMENDATION: Use Monte Carlo for ETF-based portfolio projection (confidence bands).")
    print("                Use Historical Backtesting for individual stock selection and scoring.")
elif ratio < 1.2:
    print("Monte Carlo performs similarly for both ETFs and stocks — neither clearly dominates.")
    print("RECOMMENDATION: Use Historical Backtesting as the primary ranking signal for all assets,")
    print("                and Monte Carlo only for forward-looking probability estimation.")
else:
    print("Monte Carlo performs WORSE for ETFs than stocks (unusual result — check data quality).")
    print("RECOMMENDATION: Rely primarily on Historical Backtesting for all asset scoring.")

print(f"\n(ETF/Stock MSE ratio = {ratio:.3f})")


## 6. Generative Historical Backtesting — Refactored & Parallelised

Uses the flexible `backtest_portfolio(user_config, portfolio_composition,
asset_daily_returns, baseline_daily_returns)` API.

**Training label**: `signed_squared_relative_delta = sign(Δ) × Δ²`
where `Δ = (pers_terminal − base_terminal) / |base_terminal|`.
This is the loss function used to train the two-tower recommendation model.

**Stressed configs**: goals ≥ 40% of starting capital so allocation strategy actually matters.

**Fine-grained boundary sweep**: zooms in on the 4–5yr / 11–13yr sweet spot
identified in the previous run, plus ultra-conservative and ultra-aggressive controls.


In [ ]:

import sys, os, json, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
import yfinance as yf

# ── Import and Refresh worker module ──────────────────────────────────────────
sys.path.insert(0, os.path.abspath('.'))
import _sim_worker
importlib.reload(_sim_worker)
from _sim_worker import simulate_batch, SimResult, backtest_portfolio

RESULTS_CSV   = 'debt_model_backtest_v2.csv'
N_PROFILES    = 1000
YEAR_STEP     = 252
TRADING_DAYS  = 252  # Define this up top to avoid NameErrors

# ── Fine-grained boundary configs ─────────────────────────────────────────────
BOUNDARY_CONFIGS = {
    "b4-11_r50"  : ((4, 11), 0.50),
    "b4-11_r70"  : ((4, 11), 0.70),
    "b4-12_r50"  : ((4, 12), 0.50),
    "b4-12_r70"  : ((4, 12), 0.70),
    "b4-13_r50"  : ((4, 13), 0.50),
    "b4-13_r70"  : ((4, 13), 0.70),
    "b5-12_r50"  : ((5, 12), 0.50),
    "b5-12_r70"  : ((5, 12), 0.70),
    "b3-10_r50"  : ((3, 10), 0.50),
    "b3-10_r70"  : ((3, 10), 0.70),
    "ultra_cons" : ((2, 10), 0.50),
    "ultra_agg"  : ((4, 12), 0.80),
    "current"    : ((5, 15), 0.70),
}

# ── Stressed config generator ─────────────────────────────────────────────────

# ── Grid Search Profile Generator ──────────────────────────────────────────────
# Tests combinations of (Start Capital) and (Income) vs (Financial Goals)
# Ratios: 0%, 25%, 50%, 100%, 150%, 200% of Total Goals

def generate_grid_profiles():
    profiles = []
    base_goal_amount = 100_000  # Abstract base amount for scaling
    
    # Ratios to test for Capital and Income vs Total Goals
    ratio_steps = [0.0, 0.25, 0.50, 1.0, 1.5, 2.0]
    
    # Combinations of Goal distributions (Short: Yr 3, Med: Yr 10, Long: Yr 20)
    # Represented as fractional weights
    goal_distributions = [
        {"short": 1.0, "med": 0.0, "long": 0.0},
        {"short": 0.0, "med": 1.0, "long": 0.0},
        {"short": 0.0, "med": 0.0, "long": 1.0},
        {"short": 0.5, "med": 0.5, "long": 0.0},
        {"short": 0.5, "med": 0.0, "long": 0.5},
        {"short": 0.0, "med": 0.5, "long": 0.5},
        {"short": 0.34, "med": 0.33, "long": 0.33},
        {"short": 0.80, "med": 0.10, "long": 0.10}, 
        {"short": 0.10, "med": 0.80, "long": 0.10},
        {"short": 0.10, "med": 0.10, "long": 0.80},
    ]
    
    for dist in goal_distributions:
        goals = {}
        if dist["short"] > 0: goals[3] = int(base_goal_amount * dist["short"])
        if dist["med"] > 0:   goals[10] = int(base_goal_amount * dist["med"])
        if dist["long"] > 0:  goals[20] = int(base_goal_amount * dist["long"])
        
        for cap_ratio in ratio_steps:
            for inc_ratio in ratio_steps:
                start_cap = int(base_goal_amount * cap_ratio)
                
                # Income ratio is total lifetime contribution / total goals
                # Total lifetime contribution = monthly_c * 12 * 30
                total_lifetime_inc = base_goal_amount * inc_ratio
                monthly_c = int(total_lifetime_inc / (12 * 30))
                
                profiles.append({
                    "start_cap": start_cap,
                    "monthly_contrib": monthly_c,
                    "goals": goals.copy(),
                    "meta_cap_ratio": cap_ratio,
                    "meta_inc_ratio": inc_ratio,
                    "meta_dist": str(dist)
                })
    return profiles

if os.path.exists(RESULTS_CSV):
    print(f"Found cached {RESULTS_CSV} — loading...")
    results_df = pd.read_csv(RESULTS_CSV)
else:
    print("Downloading daily market data (1994–present)...")
    hist = yf.download(['^GSPC', 'VBTIX'], start='1994-01-01', progress=False)
    prices = hist['Close'] if isinstance(hist.columns, pd.MultiIndex) else hist
    rets   = prices.pct_change().dropna(how='all').fillna(0)

    sp_ret_np   = rets['^GSPC'].values.astype(np.float64)
    bond_ret_np = (rets['VBTIX'].values.astype(np.float64) if 'VBTIX' in rets.columns else np.full(len(sp_ret_np), 0.00015))

    print(f"Generating {len(all_configs)} grid-search user configurations...")
    all_configs = generate_grid_profiles()

    max_start     = max(1, len(sp_ret_np) - int(TRADING_DAYS * 15))
    start_indices = list(range(0, max_start, YEAR_STEP))
    print(f"Historical windows: {len(start_indices)}")
    
    nested = Parallel(n_jobs=-1, backend='loky', verbose=5)(
        delayed(simulate_batch)(si, sp_ret_np, bond_ret_np, all_configs, BOUNDARY_CONFIGS)
        for si in start_indices
    )

    flat = [row for batch in nested for row in batch]
    results_df = pd.DataFrame(flat)
    date_map = {i: rets.index[i].date() for i in start_indices}
    results_df['start_date'] = results_df['start_i'].map(date_map)
    results_df.to_csv(RESULTS_CSV, index=False)

# ── Leaderboard and Visuals ───────────────────────────────────────────────────
print(f"\n=== STRATEGY LEADERBOARD ({len(results_df):,} scenarios) ===\n")
base_fail = (results_df['base_failures'] > 0).mean() * 100
base_util = results_df['base_utility'].mean()
base_term = results_df['base_terminal'].mean()

print(f"Baseline S&P500  —  Fail: {base_fail:.1f}%  |  Avg Utility: {base_util:.3f}  |  Avg Terminal: ${base_term:,.0f}\n")

rows = []
for strat, grp in results_df.groupby('strategy'):
    fail_rate = (grp['pers_failures'] > 0).mean() * 100
    rows.append({
        'Strategy': strat, 'Fail Rate': f"{fail_rate:.1f}%", 'Δ Fail pp': f"{fail_rate - base_fail:+.1f}pp",
        'Avg Utility': f"{grp['pers_utility'].mean():.3f}", 'Util Δ': f"{grp['pers_utility'].mean() - base_util:+.4f}",
        'Avg Label': f"{grp['label'].mean():+.4f}", 'Avg Terminal': f"${grp['pers_terminal'].mean():,.0f}",
    })

print(pd.DataFrame(rows).sort_values('Avg Label', ascending=False).to_string(index=False))

# ── Distribution plots ────────────────────────────────────────────────────────
top5 = pd.DataFrame(rows).sort_values('Avg Label', ascending=False).head(5)['Strategy'].tolist()
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
data_label = [results_df[results_df['strategy']==s]['label'].clip(-10, 10) for s in top5] + [pd.Series(np.zeros(len(results_df)))]
axes[0].boxplot(data_label, labels=top5+['Baseline'], patch_artist=True)
axes[0].axhline(0, color='red', linestyle='--', alpha=0.6)
axes[0].set_title('Signed Squared Relative Delta Distribution (Training Label)')
axes[0].tick_params(axis='x', rotation=30)
data_term = [results_df[results_df['strategy']==s]['pers_terminal'].clip(-5e5, 8e6) for s in top5] + [results_df['base_terminal'].clip(-5e5, 8e6)]
axes[1].boxplot(data_term, labels=top5+['Baseline'], patch_artist=True)
axes[1].set_title('Terminal Portfolio Value Distribution')
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

print(f"\n=== LABEL QUALITY CHECK ===\nLabel range: [{results_df['label'].min():.4f}, {results_df['label'].max():.4f}] | Std: {results_df['label'].std():.4f}")



## 6b. Grid Search Analysis (Income & Capital Ratios vs Performance)

Analyzes the multi-dimensional grid search caching generated in Section 6.
We explore how the starting capital ratio and income ratio impact both failure
rates and terminal utility, producing a detailed map of financial stress points.

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

RESULTS_CSV = 'debt_model_backtest_v2.csv'

try:
    df = pd.read_csv(RESULTS_CSV)
    
    if 'meta_cap_ratio' not in df.columns:
        print("meta_cap_ratio not found in results. Please re-run Section 6 to generate grid-search outputs!")
    else:
        # Group by grid constraints to see average performance
        grid_summary = df.groupby(['meta_cap_ratio', 'meta_inc_ratio']).agg(
            fail_rate=pd.NamedAgg(column='pers_failures', aggfunc=lambda x: (x > 0).mean() * 100),
            avg_utility=pd.NamedAgg(column='pers_utility', aggfunc='mean'),
            avg_label=pd.NamedAgg(column='label', aggfunc='mean'),
            avg_terminal=pd.NamedAgg(column='pers_terminal', aggfunc='mean')
        ).reset_index()

        # Print the fully sorted list of scenarios
        print("=== GRID SEARCH OUTCOMES (Sorted by Highest Utility / Least Stress) ===\n")
        sorted_grid = grid_summary.sort_values(by=['avg_utility', 'fail_rate'], ascending=[False, True])
        
        print(f"{'Cap Ratio':<10} {'Inc Ratio':<10} {'Fail Rate':<10} {'Avg Utility':<15} {'Avg Terminal $':<16}")
        print("-" * 65)
        for _, row in sorted_grid.iterrows():
            print(f"{row['meta_cap_ratio']:<10.0%} {row['meta_inc_ratio']:<10.0%} {row['fail_rate']:<9.1f}% "
                  f"{row['avg_utility']:<15.3f} ${row['avg_terminal']:,.0f}")
        
        # Plotting heatmaps
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        
        # Heatmap 1: Utility
        pivot_util = grid_summary.pivot(index="meta_cap_ratio", columns="meta_inc_ratio", values="avg_utility")
        # Format axes as percentages
        pivot_util.index = [f"{x:.0%}" for x in pivot_util.index]
        pivot_util.columns = [f"{x:.0%}" for x in pivot_util.columns]
        
        sns.heatmap(pivot_util, annot=True, fmt=".2f", cmap="YlGnBu", ax=axes[0])
        axes[0].invert_yaxis()  # So 0% is at bottom
        axes[0].set_title("Average Portfolio Utility by Grid Scenario")
        axes[0].set_ylabel("Start Capital (as % of Total Goals)")
        axes[0].set_xlabel("Lifetime Income (as % of Total Goals)")
        
        # Heatmap 2: Failure Rate
        pivot_fail = grid_summary.pivot(index="meta_cap_ratio", columns="meta_inc_ratio", values="fail_rate")
        pivot_fail.index = [f"{x:.0%}" for x in pivot_fail.index]
        pivot_fail.columns = [f"{x:.0%}" for x in pivot_fail.columns]
        
        sns.heatmap(pivot_fail, annot=True, fmt=".1f", cmap="Reds", ax=axes[1])
        axes[1].invert_yaxis()
        axes[1].set_title("Portfolio Failure Rate (%) by Grid Scenario")
        axes[1].set_ylabel("Start Capital (as % of Total Goals)")
        axes[1].set_xlabel("Lifetime Income (as % of Total Goals)")
        
        plt.tight_layout()
        plt.show()

except FileNotFoundError:
    print(f"File {RESULTS_CSV} not found. Please run Section 6 first.")


## 7. Portfolio Candidate Comparison

We build **15 diverse portfolio candidates** from the `scoring_df` asset universe
(top assets by Sharpe, return, low variance, sector focus, etc.), then run each
through the historical backtester against a representative stressed user.

**Top 5 vs Bottom 5 trajectories** plotted against the S\&P 500 baseline show
exactly how much the right portfolio composition matters given goal constraints.


In [ ]:

import sys, os, importlib
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import yfinance as yf

matplotlib.rcParams['figure.facecolor'] = 'white'
matplotlib.rcParams['axes.facecolor']   = 'white'

sys.path.insert(0, os.path.abspath('.'))
import _sim_worker
importlib.reload(_sim_worker)
from _sim_worker import (blend_portfolio_returns, allocate, backtest_portfolio,
                          _run_sim_with_trajectory, TRADING_DAYS, SIM_YEARS)

# ── Require scoring_df from Section 4b ───────────────────────────────────────
if 'scoring_df' not in dir():
    raise RuntimeError("Run Section 4b first to build scoring_df.")

# ── Representative user (moderately stressed: clear short + long goals) ───────
# -- Stressed User Configuration --
REP_USER = {
    "start_cap":       40_000,
    "monthly_contrib": 1_000,
    "goals": {3: 30_000, 8: 60_000, 20: 300_000},
}

# ── Clean scoring_df of 'Data Ghosts' (Capped annual returns at 200%) ────────
scoring_df['weighted_annual_return'] = scoring_df['weighted_annual_return'].clip(upper=2.0)

# ── Historical start: 1994 (index 0) — gives full 30-yr backtest ─────────────
HIST_START = 0

# ── Build candidate portfolios from scoring_df ────────────────────────────────
def top_tickers(df, col, n=8, ascending=False):
    return df[col].dropna().sort_values(ascending=ascending).head(n).index.tolist()

candidates = {
    "SP500 Baseline":     {"^GSPC": 1.0},
    "Top Sharpe":         {t: 1/8 for t in top_tickers(scoring_df, 'raw_sharpe', 8)},
    "Top Return":         {t: 1/8 for t in top_tickers(scoring_df, 'weighted_annual_return', 8)},
    "Low Variance":       {t: 1/8 for t in top_tickers(scoring_df, 'weighted_annual_variance', 8, ascending=True)},
    "Equal Weight All":   {t: 1/len(scoring_df) for t in scoring_df.index[:50]},
    "60/40 Index":        {"^GSPC": 0.60, "VBTIX": 0.40},
    "80/20 Growth":       {"^GSPC": 0.80, "VBTIX": 0.20},
    "40/60 Conservative": {"^GSPC": 0.40, "VBTIX": 0.60},
    "100% Bonds":         {"VBTIX": 1.0},
}

# Add sector-focused portfolios if master_research_df exists
if 'master_research_df' in dir() and 'sector' in master_research_df.columns:
    sectors = master_research_df['sector'].dropna().unique()
    for sector in ['Technology', 'Healthcare', 'Finance', 'Consumer Cyclical']:
        if sector not in sectors:
            continue
        tickers = master_research_df[master_research_df['sector'] == sector].index
        tickers = [t for t in tickers if t in scoring_df.index][:6]
        if len(tickers) >= 3:
            candidates[f"{sector[:6]} Focus"] = {t: 1/len(tickers) for t in tickers}

print(f"Portfolio candidates to evaluate: {list(candidates.keys())}")

# ── Download daily return data for all candidate tickers ─────────────────────
all_tickers = sorted(set(t for comp in candidates.values() for t in comp))
print(f"\nDownloading returns for {len(all_tickers)} tickers (1994–present)...")
raw = yf.download(all_tickers, start='1994-01-01', progress=False)

if isinstance(raw.columns, pd.MultiIndex):
    prices = raw['Close']
else:
    prices = raw

rets      = prices.pct_change().dropna(how='all').fillna(0)
n_total   = len(rets)
asset_rets = {t: rets[t].values.astype(np.float64) for t in rets.columns if t in all_tickers}
print(f"Got {n_total} trading days for {len(asset_rets)} tickers.")

# ── Run trajectory simulation for each candidate ──────────────────────────────
BOUNDARY  = (5, 15)   # (short_max_yr, medium_max_yr)
N_DAYS    = TRADING_DAYS * SIM_YEARS
cash_a, growth_a = allocate(
    REP_USER["start_cap"], REP_USER["goals"], BOUNDARY[0], BOUNDARY[1]
)

trajectory_results = {}
terminal_results   = {}

for name, composition in candidates.items():
    blended = blend_portfolio_returns(composition, asset_rets, n_total)

    # For baseline use 100% SP500, same starting capital, no bucket split
    if name == "SP500 Baseline":
        ca, ga = 0.0, float(REP_USER["start_cap"])
    else:
        ca, ga = cash_a, growth_a

    term, fails, snaps = _run_sim_with_trajectory(
        HIST_START, N_DAYS, blended, REP_USER, ca, ga
    )
    trajectory_results[name] = snaps
    terminal_results[name]   = term
    print(f"  {name:<22s}  Terminal: ${term:>12,.0f}  Failures: {fails}")

# ── Rank by terminal value ────────────────────────────────────────────────────
ranked     = sorted(terminal_results.items(), key=lambda x: x[1], reverse=True)
baseline_t = terminal_results["SP500 Baseline"]

print("\n── Ranking (terminal value) ──────────────────────────────────────────")
for rank, (name, t) in enumerate(ranked, 1):
    delta = (t - baseline_t) / abs(baseline_t) * 100
    print(f"  #{rank:2d}  {name:<22s}  ${t:>12,.0f}   ({delta:+.1f}% vs S&P500)")

# ── Identify top 5 and bottom 5 (excluding baseline) ─────────────────────────
non_base   = [(n, t) for n, t in ranked if n != "SP500 Baseline"]
top5_names = [n for n, _ in non_base[:5]]
bot5_names = [n for n, _ in non_base[-5:]]

# ── Plotting ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7), sharey=False,
                          facecolor='white')

def plot_traj(ax, names, title, cmap_name):
    cmap   = cm.get_cmap(cmap_name, len(names) + 1)
    # Baseline
    yrs_b, vals_b = zip(*trajectory_results["SP500 Baseline"])
    ax.plot(yrs_b, [v / 1e6 for v in vals_b],
            color='black', linewidth=2.5, linestyle='--',
            label=f"S&P500 Baseline  (${baseline_t/1e6:.2f}M)", zorder=10)
    # Mark goal events
    for yr, amt in REP_USER["goals"].items():
        ax.axvline(yr, color='gray', linestyle=':', alpha=0.4)
        ax.text(yr + 0.2, ax.get_ylim()[1] * 0.05 if ax.get_ylim()[1] != 0 else 0,
                f"Goal\n${amt/1000:.0f}k", fontsize=7, color='gray')

    for idx, name in enumerate(names):
        yrs, vals = zip(*trajectory_results[name])
        term      = terminal_results[name]
        delta_pct = (term - baseline_t) / abs(baseline_t) * 100
        ax.plot(yrs, [v / 1e6 for v in vals], color=cmap(idx),
                linewidth=1.8, label=f"{name}  (${term/1e6:.2f}M, {delta_pct:+.1f}%)")

    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel("Year", fontsize=11)
    ax.set_ylabel("Portfolio Value ($M)", fontsize=11)
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(True, alpha=0.3)
    ax.set_facecolor('white')

plot_traj(axes[0], top5_names, "Top 5 Portfolio Candidates vs S&P500 Baseline", "Blues")
plot_traj(axes[1], bot5_names, "Bottom 5 Portfolio Candidates vs S&P500 Baseline", "Reds")

fig.suptitle(
    f"30-Year Portfolio Trajectory Comparison\n"
    f"Representative User: ${REP_USER['start_cap']/1e3:.0f}k start | "
    f"${REP_USER['monthly_contrib']:,}/mo | "
    f"Goals at Yr {list(REP_USER['goals'].keys())}",
    fontsize=14, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.show()

# ── Summary table ─────────────────────────────────────────────────────────────
summary_rows = []
for name, term in ranked:
    delta = (term - baseline_t) / abs(baseline_t) * 100
    summary_rows.append({
        'Portfolio':       name,
        'Terminal Value':  f"${term:,.0f}",
        'vs S&P500 ($)':   f"{term - baseline_t:+,.0f}",
        'vs S&P500 (%)':   f"{delta:+.1f}%",
        'Group':           ('BASELINE' if name == 'SP500 Baseline'
                            else 'TOP 5' if name in top5_names
                            else 'BOTTOM 5' if name in bot5_names else 'MID'),
    })

print("\n" + pd.DataFrame(summary_rows).to_string(index=False))

